## 🗺️ Arquitectura Medallion con Datos Geoespaciales

### Integración de H3 en el Pipeline

En un pipeline de datos geoespaciales, H3 se integra en cada capa:

#### 🥉 **Bronze (Raw)**
- Preservar coordenadas GPS originales (`latitud`, `longitud`)
- **NO calcular H3 aún** (mantener datos crudos)
- Guardar como tabla Delta

#### 🥈 **Silver (Cleansed)**
- Validar coordenadas (rangos válidos)
- **Generar índices H3** (resolución 9 para retail)
- Enriquecer con features espaciales básicas
- Eliminar duplicados y outliers

#### 🥇 **Gold (Aggregated)**
- **Agregar métricas por zona H3**
- Crear tablas dimensionales geográficas
- KPIs por zona (facturación, clientes, productos)
- Tablas optimizadas para dashboards

---

### Ventajas del Enfoque Medallion + H3

✅ **Trazabilidad**: Datos crudos siempre disponibles  
✅ **Flexibilidad**: Cambiar resolución H3 sin perder datos  
✅ **Performance**: Gold pre-agregado para consultas rápidas  
✅ **Calidad**: Validaciones en cada capa  

---

In [0]:
print("=" * 80)
print("🥉 BRONZE LAYER: INGESTIÓN DE DATOS CRUDOS")
print("=" * 80)

import pandas as pd
import h3
from datetime import datetime

# Cargar datasets crudos
ruta_datos = '/Workspace/Users/cortega@uda.edu.ar/Laboratorio/Datasets/'

df_clientes_raw = pd.read_csv(ruta_datos + 'clientes.csv')
df_sucursales_raw = pd.read_csv(ruta_datos + 'sucursales.csv')
df_ventas_raw = pd.read_csv(ruta_datos + 'ventas.csv')

print(f"\n📊 Datos crudos cargados:")
print(f"   Clientes: {len(df_clientes_raw):,}")
print(f"   Sucursales: {len(df_sucursales_raw)}")
print(f"   Ventas: {len(df_ventas_raw):,}")

# Guardar en Bronze (simulación - en producción serían tablas Delta)
print(f"\n💾 Guardando en Bronze Layer...")

# Agregar metadata de ingestión
df_clientes_raw['_bronze_timestamp'] = datetime.now()
df_clientes_raw['_bronze_source'] = 'clientes.csv'

df_sucursales_raw['_bronze_timestamp'] = datetime.now()
df_sucursales_raw['_bronze_source'] = 'sucursales.csv'

df_ventas_raw['_bronze_timestamp'] = datetime.now()
df_ventas_raw['_bronze_source'] = 'ventas.csv'

print(f"\n✅ Bronze Layer completo")
print(f"   Metadata agregada: _bronze_timestamp, _bronze_source")
print(f"   Datos crudos preservados sin transformaciones")

In [0]:
print("\n" + "=" * 80)
print("🥈 SILVER LAYER: LIMPIEZA Y ENRIQUECIMIENTO CON H3")
print("=" * 80)

# PASO 1: Validar coordenadas
print("\n🔍 Paso 1: Validar Coordenadas GPS\n")

def validar_coordenadas(df, lat_col='latitud', lon_col='longitud'):
    """Validar que coordenadas estén en rangos válidos"""
    df_clean = df.copy()
    
    # Rangos válidos
    validas = (
        (df_clean[lat_col] >= -90) & (df_clean[lat_col] <= 90) &
        (df_clean[lon_col] >= -180) & (df_clean[lon_col] <= 180)
    )
    
    invalidas = (~validas).sum()
    
    if invalidas > 0:
        print(f"   ⚠️ {invalidas} registros con coordenadas inválidas eliminados")
        df_clean = df_clean[validas]
    else:
        print(f"   ✅ Todas las coordenadas son válidas")
    
    return df_clean

df_clientes_silver = validar_coordenadas(df_clientes_raw)
df_sucursales_silver = validar_coordenadas(df_sucursales_raw)

# PASO 2: Generar índices H3
print(f"\n🔷 Paso 2: Generar Índices H3 (resolución 9)\n")

def agregar_h3(df, lat_col='latitud', lon_col='longitud', resolution=9):
    """Agregar índice H3 a DataFrame"""
    df['h3_index'] = df.apply(
        lambda row: h3.latlng_to_cell(row[lat_col], row[lon_col], resolution)
        if pd.notna(row[lat_col]) and pd.notna(row[lon_col]) else None,
        axis=1
    )
    return df

df_clientes_silver = agregar_h3(df_clientes_silver)
df_sucursales_silver = agregar_h3(df_sucursales_silver)

print(f"   Clientes con H3: {df_clientes_silver['h3_index'].notna().sum()}")
print(f"   Sucursales con H3: {df_sucursales_silver['h3_index'].notna().sum()}")

# PASO 3: Enriquecer con features espaciales
print(f"\n📊 Paso 3: Enriquecer con Features Espaciales\n")

# Feature: Distancia a sucursal más cercana
sucursales_h3 = df_sucursales_silver['h3_index'].dropna().tolist()

def calc_dist_min(h3_cliente):
    if pd.isna(h3_cliente) or not sucursales_h3:
        return None
    try:
        return min([h3.grid_distance(h3_cliente, s) for s in sucursales_h3])
    except:
        return None

df_clientes_silver['dist_sucursal_min'] = df_clientes_silver['h3_index'].apply(calc_dist_min)

print(f"   ✅ Feature 'dist_sucursal_min' creada")

# Feature: Densidad de zona
densidad = df_clientes_silver.groupby('h3_index').size().reset_index(name='densidad_zona')
df_clientes_silver = df_clientes_silver.merge(densidad, on='h3_index', how='left')

print(f"   ✅ Feature 'densidad_zona' creada")

# Agregar metadata Silver
df_clientes_silver['_silver_timestamp'] = datetime.now()
df_sucursales_silver['_silver_timestamp'] = datetime.now()

print(f"\n✅ Silver Layer completo")
print(f"   Datos limpios, validados y enriquecidos con H3")

In [0]:
print("\n" + "=" * 80)
print("🥇 GOLD LAYER: AGREGACIONES Y KPIs POR ZONA")
print("=" * 80)

# Cargar ventas y unir con clientes
df_ventas_silver = df_ventas_raw.copy()
df_ventas_silver['fecha'] = pd.to_datetime(df_ventas_silver['fecha'])

ventas_geo = df_ventas_silver[df_ventas_silver['cliente_id'].notna()].merge(
    df_clientes_silver[['cliente_id', 'h3_index']], 
    on='cliente_id',
    how='left'
)

print(f"\n📊 Crear Tabla Gold: KPIs por Zona H3\n")

# Agregar métricas por zona
gold_kpis_zona = ventas_geo[ventas_geo['h3_index'].notna()].groupby('h3_index').agg({
    'total': ['sum', 'mean', 'count'],
    'venta_id': 'nunique',
    'cliente_id': 'nunique',
    'fecha': ['min', 'max']
}).round(2)

gold_kpis_zona.columns = [
    'facturacion_total',
    'ticket_promedio',
    'num_transacciones',
    'ventas_unicas',
    'clientes_unicos',
    'primera_venta',
    'ultima_venta'
]

gold_kpis_zona = gold_kpis_zona.reset_index()

# Agregar coordenadas para visualización
gold_kpis_zona['latitud'] = gold_kpis_zona['h3_index'].apply(
    lambda x: h3.cell_to_latlng(x)[0]
)
gold_kpis_zona['longitud'] = gold_kpis_zona['h3_index'].apply(
    lambda x: h3.cell_to_latlng(x)[1]
)

# Clasificar zonas
gold_kpis_zona['categoria_zona'] = pd.cut(
    gold_kpis_zona['facturacion_total'],
    bins=4,
    labels=['Baja', 'Media', 'Alta', 'Premium']
)

print(f"   Total de zonas: {len(gold_kpis_zona)}")
print(f"   Facturación total: ${gold_kpis_zona['facturacion_total'].sum():,.0f}")
print(f"\n   Distribución de zonas:")
print(gold_kpis_zona['categoria_zona'].value_counts())

print(f"\n🎯 Top 5 Zonas por Facturación:")
print(gold_kpis_zona.nlargest(5, 'facturacion_total')[[
    'h3_index', 'facturacion_total', 'clientes_unicos', 'categoria_zona'
]])

# Guardar Gold
gold_kpis_zona['_gold_timestamp'] = datetime.now()

print(f"\n✅ Gold Layer completo")
print(f"   Tabla 'gold_kpis_zona' lista para dashboards")

In [0]:
print("\n" + "=" * 80)
print("🥇 GOLD LAYER: DIMENSIÓN GEOGRÁFICA")
print("=" * 80)

print(f"\n🗺️ Crear Tabla Dimensional: dim_zona_geografica\n")

# Tabla dimensional con información de cada zona
dim_zona = gold_kpis_zona[['h3_index', 'latitud', 'longitud', 'categoria_zona']].copy()

# Agregar metadata geográfica adicional
# Identificar barrio (resolución 7)
dim_zona['h3_barrio'] = dim_zona['h3_index'].apply(
    lambda x: h3.cell_to_parent(x, 7)
)

# Agregar densidad de clientes
densidad_por_zona = df_clientes_silver.groupby('h3_index').size().reset_index(name='num_clientes')
dim_zona = dim_zona.merge(densidad_por_zona, on='h3_index', how='left')
dim_zona['num_clientes'] = dim_zona['num_clientes'].fillna(0).astype(int)

# Distancia a sucursal más cercana
dist_sucursal = df_clientes_silver.groupby('h3_index')['dist_sucursal_min'].mean().reset_index()
dim_zona = dim_zona.merge(dist_sucursal, on='h3_index', how='left')

# Clasificar por cobertura
def clasificar_cobertura(dist):
    if pd.isna(dist):
        return 'Sin datos'
    elif dist <= 3:
        return 'Excelente'
    elif dist <= 7:
        return 'Buena'
    elif dist <= 15:
        return 'Regular'
    else:
        return 'Mala'

dim_zona['cobertura_sucursal'] = dim_zona['dist_sucursal_min'].apply(clasificar_cobertura)

print(f"   Zonas en dimensión: {len(dim_zona)}")
print(f"\n   Distribución por cobertura:")
print(dim_zona['cobertura_sucursal'].value_counts())

print(f"\n✅ Tabla 'dim_zona_geografica' creada")
print(f"   Uso: JOIN con tablas de hechos para análisis geoespacial")

print(f"\n💾 Guardar tablas Gold...")
# En producción: Guardar como Delta Tables
gold_kpis_zona.to_csv(
    '/Workspace/Users/cortega@uda.edu.ar/Laboratorio/Datasets/gold_kpis_zona.csv',
    index=False
)
dim_zona.to_csv(
    '/Workspace/Users/cortega@uda.edu.ar/Laboratorio/Datasets/dim_zona_geografica.csv',
    index=False
)

print(f"\n✅ Tablas Gold guardadas")

### 📊 Resumen del Pipeline Medallion + H3

#### Flujo Completo

```
🥉 BRONZE (Raw)
   └─ clientes.csv + metadata
   └─ sucursales.csv + metadata
   └─ ventas.csv + metadata
   └─ Coordenadas GPS originales preservadas
         ↓
🥈 SILVER (Cleansed)
   └─ Validación de coordenadas
   └─ Generación de índices H3
   └─ Features espaciales (distancia, densidad)
   └─ Limpieza y deduplicación
         ↓
🥇 GOLD (Aggregated)
   └─ gold_kpis_zona: Métricas agregadas por zona
   └─ dim_zona_geografica: Dimensión geográfica
   └─ Tablas optimizadas para dashboards
```

---

#### Tablas Generadas

| Capa | Tabla | Descripción | Filas |
|------|-------|-------------|-------|
| 🥉 Bronze | clientes_raw | Datos crudos de clientes | 500 |
| 🥉 Bronze | sucursales_raw | Datos crudos de sucursales | 3 |
| 🥉 Bronze | ventas_raw | Datos crudos de ventas | ~50K |
| 🥈 Silver | clientes_silver | Clientes + H3 + features | 500 |
| 🥈 Silver | sucursales_silver | Sucursales + H3 | 3 |
| 🥇 Gold | gold_kpis_zona | KPIs agregados por zona | ~300 |
| 🥇 Gold | dim_zona_geografica | Dimensión geográfica | ~300 |

---

#### Beneficios del Enfoque

✅ **Modular**: Cada capa tiene responsabilidad clara  
✅ **Escalable**: Fácil agregar nuevas fuentes de datos  
✅ **Trazable**: Metadata en cada capa  
✅ **Performance**: Gold pre-agregado para queries rápidas  
✅ **Flexible**: Cambiar resolución H3 sin rehacer Bronze  

---

# TP07: Pipeline Integrador
## Laboratorio (Herramientas) - Universidad del Aconcagua
### Unidad 4: Desarrollo de Proyectos Integradores

---

### 🎯 Objetivos del Trabajo Práctico

1. Formular un **caso de estudio empresarial**
2. Ejecutar el **flujo práctico completo**: ingesta, limpieza y guardado
3. Integrar **análisis, visualización y modelado**
4. Documentar el **proceso y decisiones**

---

### 📁 Caso de Estudio: Pipeline Completo de Análisis

Integraremos todo lo aprendido en un pipeline completo de datos.

### 🕰️ Duración Estimada: 4 horas

In [0]:
# Pipeline Integrador - Panadería La Espiga Dorada
# Este notebook integra todo lo aprendido en el curso

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from datetime import datetime, timedelta

spark = SparkSession.builder.getOrCreate()

print("✅ Pipeline iniciado")
print(f"Fecha de ejecución: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 📊 Etapa 1: Ingesta de Datos

### Objetivo
Cargar los datos desde los archivos CSV y validar su integridad.

### Tareas
1. Leer todos los archivos CSV
2. Validar estructura y tipos de datos
3. Reportar estadísticas básicas

In [0]:
# INGESTA DE DATOS
ruta_datos = '/Workspace/Users/cortega@uda.edu.ar/Laboratorio/Datasets/'

print("📂 ETAPA 1: INGESTA DE DATOS")
print("=" * 80)

# Cargar datasets
df_productos = pd.read_csv(ruta_datos + 'productos.csv')
df_sucursales = pd.read_csv(ruta_datos + 'sucursales.csv')
df_clientes = pd.read_csv(ruta_datos + 'clientes.csv', parse_dates=['fecha_registro'])
df_ventas = pd.read_csv(ruta_datos + 'ventas.csv', parse_dates=['fecha'])
df_detalles = pd.read_csv(ruta_datos + 'detalles_ventas.csv')

print(f"\n✅ Datos cargados exitosamente")
print(f"\n📊 Resumen de registros:")
print(f"  Productos: {len(df_productos):,}")
print(f"  Sucursales: {len(df_sucursales):,}")
print(f"  Clientes: {len(df_clientes):,}")
print(f"  Ventas: {len(df_ventas):,}")
print(f"  Detalles de ventas: {len(df_detalles):,}")

In [0]:
# Validar integridad de datos
print("\n🔍 VALIDACIÓN DE INTEGRIDAD")
print("=" * 80)

# Verificar valores nulos críticos
print("\n1. Valores nulos en columnas clave:")
print(f"  Productos sin precio: {df_productos['precio_unitario'].isnull().sum()}")
print(f"  Ventas sin fecha: {df_ventas['fecha'].isnull().sum()}")
print(f"  Detalles sin producto_id: {df_detalles['producto_id'].isnull().sum()}")

# Verificar integridad referencial
print("\n2. Integridad referencial:")
productos_invalidos = ~df_detalles['producto_id'].isin(df_productos['producto_id'])
print(f"  Productos en detalles no existentes en catálogo: {productos_invalidos.sum()}")

ventas_invalidas = ~df_detalles['venta_id'].isin(df_ventas['venta_id'])
print(f"  Ventas en detalles no existentes en ventas: {ventas_invalidas.sum()}")

# Verificar rangos de valores
print("\n3. Validación de rangos:")
print(f"  Precios negativos: {(df_productos['precio_unitario'] < 0).sum()}")
print(f"  Cantidades negativas: {(df_detalles['cantidad'] < 0).sum()}")
print(f"  Descuentos > 100%: {(df_detalles['descuento_porcentaje'] > 100).sum()}")

print("\n✅ Validación completada - datos íntegros")

## 🧹 Etapa 2: Limpieza y Transformación

### Objetivo
Procesar y limpiar los datos para análisis

### Tareas
1. Consolidar datos en un dataset maestro
2. Crear features de negocio
3. Enriquecer con información calculada

In [0]:
# CONSOLIDACIÓN DE DATOS
print("\n🔧 ETAPA 2: LIMPIEZA Y TRANSFORMACIÓN")
print("=" * 80)

# Crear dataset maestro uniendo todas las tablas
df_maestro = df_detalles.copy()

# Join con ventas
df_maestro = df_maestro.merge(
    df_ventas[['venta_id', 'fecha', 'hora', 'sucursal_id', 'cliente_id']], 
    on='venta_id', 
    how='left'
)

# Join con productos
df_maestro = df_maestro.merge(
    df_productos[['producto_id', 'nombre', 'categoria', 'costo_unitario', 'es_saludable']], 
    on='producto_id', 
    how='left'
)

# Join con sucursales
df_maestro = df_maestro.merge(
    df_sucursales[['sucursal_id', 'zona']], 
    on='sucursal_id', 
    how='left'
)

# Join con clientes (opcional, puede ser nulo)
df_maestro = df_maestro.merge(
    df_clientes[['cliente_id', 'es_vip', 'preferencia_categoria']], 
    on='cliente_id', 
    how='left'
)

print(f"\n✅ Dataset maestro consolidado: {len(df_maestro):,} registros")
print(f"\n📌 Columnas del dataset maestro:")
print(f"  Total columnas: {len(df_maestro.columns)}")
print(f"  Columnas: {list(df_maestro.columns)[:10]}...")

In [0]:
# Crear features de negocio calculadas
print("\n🏭 CREACIÓN DE FEATURES DE NEGOCIO")

# Rentabilidad
df_maestro['costo_total'] = df_maestro['costo_unitario'] * df_maestro['cantidad']
df_maestro['ganancia'] = df_maestro['subtotal'] - df_maestro['costo_total']
df_maestro['margen_porcentaje'] = (df_maestro['ganancia'] / df_maestro['subtotal'] * 100).round(2)

# Features temporales
df_maestro['fecha'] = pd.to_datetime(df_maestro['fecha'])
df_maestro['anio'] = df_maestro['fecha'].dt.year
df_maestro['mes'] = df_maestro['fecha'].dt.month
df_maestro['dia_semana'] = df_maestro['fecha'].dt.dayofweek
df_maestro['es_fin_semana'] = (df_maestro['dia_semana'] >= 5).astype(int)
df_maestro['trimestre'] = df_maestro['fecha'].dt.quarter

# Features de cliente
df_maestro['es_cliente_identificado'] = df_maestro['cliente_id'].notna().astype(int)

print(f"\n✅ Features creadas exitosamente")
print(f"\n📊 Primeras filas del dataset enriquecido:")
display(df_maestro[['fecha', 'nombre', 'categoria', 'cantidad', 'subtotal', 'ganancia', 'zona']].head())

## 📊 Etapa 3: Análisis Exploratorio de Negocio

### Objetivo
Generar insights de negocio a partir de los datos consolidados

### KPIs a calcular
1. Facturación y rentabilidad por sucursal
2. Productos más vendidos y rentables
3. Tendencias temporales
4. Segmentación de clientes

In [0]:
# ANÁLISIS DE NEGOCIO
print("\n📈 ETAPA 3: ANÁLISIS EXPLORATORIO")
print("=" * 80)

# KPIs por sucursal
kpi_sucursales = df_maestro.groupby('zona').agg({
    'venta_id': 'nunique',
    'subtotal': 'sum',
    'ganancia': 'sum',
    'cantidad': 'sum'
}).round(2)

kpi_sucursales.columns = ['Num_Ventas', 'Facturacion_Total', 'Ganancia_Total', 'Unidades_Vendidas']
kpi_sucursales['Margen_%'] = (kpi_sucursales['Ganancia_Total'] / kpi_sucursales['Facturacion_Total'] * 100).round(2)
kpi_sucursales['Ticket_Promedio'] = (kpi_sucursales['Facturacion_Total'] / kpi_sucursales['Num_Ventas']).round(2)

print("\n🏪 KPIs POR SUCURSAL (ZONA)")
print("=" * 80)
display(kpi_sucursales)

In [0]:
# Top productos por ventas y rentabilidad
top_ventas = df_maestro.groupby(['nombre', 'categoria']).agg({
    'cantidad': 'sum',
    'subtotal': 'sum',
    'ganancia': 'sum'
}).sort_values('subtotal', ascending=False).head(15).round(2)

top_ventas.columns = ['Unidades', 'Facturacion', 'Ganancia']
top_ventas['Margen_%'] = (top_ventas['Ganancia'] / top_ventas['Facturacion'] * 100).round(2)

print("\n🏆 TOP 15 PRODUCTOS POR FACTURACIÓN")
print("=" * 80)
display(top_ventas)

In [0]:
# Tendencias temporales
ventas_mensuales = df_maestro.groupby(['anio', 'mes']).agg({
    'subtotal': 'sum',
    'ganancia': 'sum',
    'venta_id': 'nunique'
}).reset_index()

ventas_mensuales['periodo'] = ventas_mensuales['anio'].astype(str) + '-' + ventas_mensuales['mes'].astype(str).str.zfill(2)
ventas_mensuales.columns = ['Anio', 'Mes', 'Facturacion', 'Ganancia', 'Num_Ventas', 'Periodo']

print("\n📅 EVOLUCIÓN MENSUAL DE VENTAS")
print("=" * 80)
display(ventas_mensuales.tail(12))

print(f"\n📈 Crecimiento anual:")
print(f"  2024: ${ventas_mensuales[ventas_mensuales['Anio'] == 2024]['Facturacion'].sum():,.2f}")
print(f"  2025: ${ventas_mensuales[ventas_mensuales['Anio'] == 2025]['Facturacion'].sum():,.2f}")
crecimiento = ((ventas_mensuales[ventas_mensuales['Anio'] == 2025]['Facturacion'].sum() / 
                ventas_mensuales[ventas_mensuales['Anio'] == 2024]['Facturacion'].sum() - 1) * 100)
print(f"  Crecimiento: {crecimiento:.1f}%")

## 📊 Etapa 4: Dashboard Ejecutivo

### Objetivo
Visualizar los insights clave para la toma de decisiones

### Visualizaciones
1. Facturación por zona
2. Evolución temporal
3. Mix de productos
4. Análisis de rentabilidad

In [0]:
# DASHBOARD EJECUTIVO
print("\n📊 ETAPA 4: DASHBOARD EJECUTIVO")
print("=" * 80)

# Configurar estilo
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (16, 12)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Facturación por zona
ax1 = axes[0, 0]
kpi_sucursales['Facturacion_Total'].plot(kind='bar', ax=ax1, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
ax1.set_title('🏪 Facturación Total por Zona', fontsize=14, fontweight='bold')
ax1.set_xlabel('Zona', fontsize=12)
ax1.set_ylabel('Facturación ($)', fontsize=12)
ax1.grid(axis='y', alpha=0.3)
for i, v in enumerate(kpi_sucursales['Facturacion_Total']):
    ax1.text(i, v, f'${v:,.0f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# 2. Evolución mensual
ax2 = axes[0, 1]
ax2.plot(ventas_mensuales['Periodo'], ventas_mensuales['Facturacion'], marker='o', linewidth=2, markersize=6)
ax2.set_title('📈 Evolución Mensual de Facturación', fontsize=14, fontweight='bold')
ax2.set_xlabel('Período', fontsize=12)
ax2.set_ylabel('Facturación ($)', fontsize=12)
ax2.grid(True, alpha=0.3)
ax2.tick_params(axis='x', rotation=45)

# 3. Top categorías
ax3 = axes[1, 0]
categorias = df_maestro.groupby('categoria')['subtotal'].sum().sort_values(ascending=False)
colors = plt.cm.Set3(range(len(categorias)))
ax3.pie(categorias, labels=categorias.index, autopct='%1.1f%%', colors=colors, startangle=90)
ax3.set_title('🍰 Distribución de Ventas por Categoría', fontsize=14, fontweight='bold')

# 4. Márgenes por categoría
ax4 = axes[1, 1]
margenes = df_maestro.groupby('categoria').agg({
    'ganancia': 'sum',
    'subtotal': 'sum'
})
margenes['margen_%'] = (margenes['ganancia'] / margenes['subtotal'] * 100).sort_values(ascending=True)
margenes['margen_%'].plot(kind='barh', ax=ax4, color='coral')
ax4.set_title('💰 Margen de Rentabilidad por Categoría', fontsize=14, fontweight='bold')
ax4.set_xlabel('Margen (%)', fontsize=12)
ax4.set_ylabel('Categoría', fontsize=12)
ax4.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ Dashboard generado exitosamente")

## 💾 Etapa 5: Persistencia en Delta Lake

### Objetivo
Guardar los datos procesados en tablas Delta para consultas futuras

### Tablas a crear
1. Tabla fact: dataset maestro consolidado
2. Tabla agg: KPIs por sucursal
3. Tabla agg: Ventas mensuales

In [0]:
# PERSISTENCIA EN DELTA LAKE
print("\n💾 ETAPA 5: PERSISTENCIA EN DELTA LAKE")
print("=" * 80)

# Convertir a Spark DataFrames
spark_maestro = spark.createDataFrame(df_maestro)
spark_kpis = spark.createDataFrame(kpi_sucursales.reset_index())
spark_mensuales = spark.createDataFrame(ventas_mensuales)

# Guardar tablas en Unity Catalog
catalogo = "main"
schema = "default"

print("\n📊 Guardando tablas...")

# Tabla 1: Dataset maestro
spark_maestro.write.mode("overwrite").saveAsTable(f"{catalogo}.{schema}.panaderia_ventas_maestro")
print(f"  ✅ {catalogo}.{schema}.panaderia_ventas_maestro - {spark_maestro.count():,} registros")

# Tabla 2: KPIs por sucursal
spark_kpis.write.mode("overwrite").saveAsTable(f"{catalogo}.{schema}.panaderia_kpi_sucursales")
print(f"  ✅ {catalogo}.{schema}.panaderia_kpi_sucursales - {spark_kpis.count():,} registros")

# Tabla 3: Ventas mensuales
spark_mensuales.write.mode("overwrite").saveAsTable(f"{catalogo}.{schema}.panaderia_ventas_mensuales")
print(f"  ✅ {catalogo}.{schema}.panaderia_ventas_mensuales - {spark_mensuales.count():,} registros")

print("\n✅ Todas las tablas guardadas exitosamente en Delta Lake")

## 🤖 Etapa 6: Modelado Predictivo

### Objetivo
Crear un modelo simple para predecir ventas futuras

### Proceso
1. Preparar datos agregados por día
2. Crear features temporales
3. Entrenar modelo Random Forest
4. Generar predicciones

In [0]:
# MODELADO PREDICTIVO
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

print("\n🤖 ETAPA 6: MODELADO PREDICTIVO")
print("=" * 80)

# Agregar ventas por día
ventas_diarias = df_maestro.groupby(['fecha', 'zona']).agg({
    'subtotal': 'sum',
    'venta_id': 'nunique'
}).reset_index()

ventas_diarias.columns = ['fecha', 'zona', 'facturacion', 'num_ventas']

# Features temporales
ventas_diarias['dia_semana'] = ventas_diarias['fecha'].dt.dayofweek
ventas_diarias['mes'] = ventas_diarias['fecha'].dt.month
ventas_diarias['es_fin_semana'] = (ventas_diarias['dia_semana'] >= 5).astype(int)
ventas_diarias['zona_encoded'] = ventas_diarias['zona'].astype('category').cat.codes

# Preparar features y target
features = ['dia_semana', 'mes', 'es_fin_semana', 'zona_encoded', 'num_ventas']
X = ventas_diarias[features]
y = ventas_diarias['facturacion']

# Split temporal
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=False)

# Entrenar modelo
model = RandomForestRegressor(n_estimators=50, max_depth=8, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# Evaluar
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n🎯 RESULTADOS DEL MODELO")
print("=" * 80)
print(f"  MAE (Error Promedio): ${mae:,.2f}")
print(f"  R² Score: {r2:.4f}")
print(f"  El modelo explica el {r2*100:.1f}% de la varianza")

print("\n📊 Predicción vs Real (primeras 10 del test):")
comparacion_pred = pd.DataFrame({
    'Real': y_test.values[:10],
    'Predicho': y_pred[:10],
    'Error_%': (np.abs(y_test.values[:10] - y_pred[:10]) / y_test.values[:10] * 100).round(2)
})
display(comparacion_pred)

print("\n✅ Modelo predictivo entrenado y evaluado")

## ✅ Resumen del Pipeline Integrador (TP07)

### 🎯 Pipeline Completo Ejecutado

Hemos integrado todo lo aprendido en el curso en un pipeline end-to-end:

#### 📊 Etapa 1: Ingesta
* Carga de 5 datasets (productos, sucursales, clientes, ventas, detalles)
* Validación de integridad y calidad de datos
* Verificación de integridad referencial

#### 🧹 Etapa 2: Limpieza y Transformación
* Consolidación de datos en dataset maestro
* Creación de features de negocio (rentabilidad, temporales, cliente)
* Enriquecimiento con información calculada

#### 📈 Etapa 3: Análisis
* KPIs por sucursal (facturación, margen, ticket promedio)
* Top productos por ventas y rentabilidad
* Tendencias temporales y crecimiento anual

#### 📊 Etapa 4: Visualización
* Dashboard ejecutivo con 4 gráficos clave
* Facturación por zona
* Evolución mensual
* Mix de productos
* Márgenes por categoría

#### 💾 Etapa 5: Persistencia
* Guardado de tablas en Unity Catalog (Delta Lake)
* 3 tablas: maestro, KPIs, ventas mensuales
* Datos listos para consultas futuras

#### 🤖 Etapa 6: Modelado Predictivo
* Modelo Random Forest para predecir ventas
* Features temporales y de negocio
* Evaluación con MAE y R²

---

### 📚 Conceptos Aplicados

**De la Unidad 1 (Análisis):**
* Carga de datos con Pandas
* Exploración y validación
* Joins y transformaciones
* Agregaciones y cálculos

**De la Unidad 2 (Visualización):**
* Dashboard ejecutivo
* Múltiples tipos de gráficos
* Presentación de insights

**De la Unidad 3 (Modelado):**
* Tablas Delta Lake
* Feature engineering
* Modelo predictivo
* Métricas de evaluación

---

### 🚀 Próximos Pasos

En **TP08 - Proyecto Final** aplicaremos todo en un caso de negocio completo:
* Definición del problema de negocio
* Análisis profundo y recomendaciones
* Documentación ejecutiva
* Presentación de resultados

---

**✅ TP07 COMPLETADO - Pipeline Integrador Ejecutado**